In [1]:
import numpy as np
import pandas as pd
import os
import glob
import itertools
import scanpy as sc
import natsort
import json

# import matplotlib.pyplot as plt
# import seaborn as sns

# from scroutines import basicu
# from scroutines import powerplots

# import scanpy.external as sce

# import snapatac2 as snap

In [2]:
ddir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_dev_merged'
outdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/astro_john'
!ls $ddir
!ls $outdir

bigtensor_cheng22_subclass_v2.json  bigtensor_yoo25_subclass_v2.npy
bigtensor_cheng22_subclass_v2.npy   cheng22_astro.h5ad
bigtensor_gao25_subclass.json	    gao25_astro.h5ad
bigtensor_gao25_subclass.npy	    yoo25_astro.h5ad
bigtensor_yoo25_subclass_v2.json
cellchat_from_riki  scenicplus_inputdata


In [3]:
# rna
adata = sc.read(os.path.join(ddir, 'yoo25_astro.h5ad'), backed='r')
print(adata)

AnnData object with n_obs × n_vars = 14028 × 16572 backed at '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_dev_merged/yoo25_astro.h5ad'
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'cond', 'biosample'
    var: 'feature_types'


In [4]:
cell_barcodes = adata.obs_names.values
cell_barcodes

array(['TCTTAGTTCAGCAAAG-1-P6b-2023 Multiome-0-0',
       'GGAACAATCTTAGGGT-1-P6c-2023 Multiome-0-0',
       'TGGCCTTTCCGTTATT-1-P6a-2023 Multiome-0-0', ...,
       'CGGACCTAGGCTTAAC-1-P21DRa-2023 Multiome-10-0',
       'GAGGGAGCATAATGAG-1-P21DRb-2023 Multiome-10-0',
       'GAGAAACGTTAGCATG-1-P21DRa-2023 Multiome-10-0'], dtype=object)

In [5]:
def streamline_barcode(x):
    return x.split(' ')[0][:-len("-2023")]

def streamline_barcode2(x):
    return '-'.join(x.split('-')[:2])

def streamline_barcode3(x):
    return x.split('-')[2]

vfunc  = np.vectorize(streamline_barcode)
vfunc2 = np.vectorize(streamline_barcode2)
vfunc3 = np.vectorize(streamline_barcode3)

cell_barcodes_simp = vfunc(cell_barcodes)
cell_barcodes_nosamp = vfunc2(cell_barcodes_simp)
cell_barcodes_samp   = vfunc3(cell_barcodes_simp)

In [6]:
uniqs, counts = np.unique(cell_barcodes_simp, return_counts=True)
np.any(counts > 1)

False

In [7]:
cell_barcodes_nosamp

array(['TCTTAGTTCAGCAAAG-1', 'GGAACAATCTTAGGGT-1', 'TGGCCTTTCCGTTATT-1',
       ..., 'CGGACCTAGGCTTAAC-1', 'GAGGGAGCATAATGAG-1',
       'GAGAAACGTTAGCATG-1'], dtype='<U18')

In [8]:
cell_barcodes_samp

array(['P6b', 'P6c', 'P6a', ..., 'P21DRa', 'P21DRb', 'P21DRa'],
      dtype='<U6')

In [9]:
uniq_samps = np.unique(cell_barcodes_samp)
uniq_samps

array(['P10a', 'P10b', 'P12DRa', 'P12DRb', 'P12a', 'P12b', 'P12c',
       'P14DRa', 'P14DRb', 'P14a', 'P14b', 'P17DRa', 'P17DRb', 'P17a',
       'P17b', 'P21DRa', 'P21DRb', 'P21a', 'P21b', 'P6a', 'P6b', 'P6c',
       'P8a', 'P8b', 'P8c'], dtype='<U6')

In [10]:
# Write barcodes to file first
for samp in uniq_samps:
    sel_cond = cell_barcodes_samp == samp
    sel_barcodes = cell_barcodes_nosamp[sel_cond]
    
    fout = f'/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_{samp}.txt'
    print(fout)
    with open(fout, "w") as f:
        f.write("\n".join(sel_barcodes))

/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_P10a.txt
/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_P10b.txt
/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_P12DRa.txt
/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_P12DRb.txt
/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_P12a.txt
/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_P12b.txt
/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_P12c.txt
/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_P14DRa.txt
/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_P14DRb.txt
/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_P14a.txt
/u/home/f/f7xiesnm/astro_john/scenicplus_i

In [11]:
# parse it by sample 